# 🏋️ Workout Video Analysis with Gemini

**How to use:** click **Runtime → Run all** (or Cmd + F9).
- If asked *"Grant access to secret?"*, click **Grant access**.
- When a **Choose Files** button appears in Step 3, pick your video.
- Read the analysis under Step 4.

Your API key is loaded from Colab **Secrets** (🔑 on the left). It is never printed or stored in this notebook.

## Step 1 · Install the Gemini SDK

In [ ]:
!pip install -q -U google-genai

## Step 2 · Load your API key and pick a model
If your secret has a different name, add it to `SECRET_NAMES` below.

In [ ]:
from google.colab import userdata
from google import genai

SECRET_NAMES = ["GOOGLE_API_KEY", "GEMINI_API_KEY", "video-clip", "video-clip-key", "video-clip-analysis"]

api_key, used = None, None
for name in SECRET_NAMES:
    try:
        api_key = userdata.get(name)
        if api_key:
            used = name
            break
    except Exception:
        pass

if not api_key:
    raise SystemExit("❌ No secret found. Open 🔑 Secrets, check the exact secret name, make sure the "
                     "'Notebook access' toggle is ON, and add the name to SECRET_NAMES above.")
if not api_key.startswith("AIza") or len(api_key) < 30:
    print(f"⚠️ Secret '{used}' doesn't look like a Gemini key (should start with 'AIza', ~39 chars). "
          "Use 🔑 Secrets → 'Gemini API keys ▾' → Import key.")

client = genai.Client(api_key=api_key)

# Pick the best available Flash model automatically
models = [m.name.replace("models/", "") for m in client.models.list()
          if "generateContent" in (m.supported_actions or [])]
flash = sorted([m for m in models if "flash" in m and "lite" not in m
                and "image" not in m and "tts" not in m and "live" not in m and "audio" not in m], reverse=True)
MODEL = next((m for m in flash if "preview" not in m and "exp" not in m), flash[0] if flash else "gemini-2.5-flash")
print(f"✅ Key loaded from secret '{used}'. Using model: {MODEL}")

## Step 3 · Upload your video
Click **Choose Files** below and select your workout video (mp4 / mov / webm).

In [ ]:
from google.colab import files
import time

uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file selected — re-run this cell and choose a video.")
video_path = next(iter(uploaded))
print("Uploading to Gemini:", video_path)

video = client.files.upload(file=video_path)
while video.state.name == "PROCESSING":
    print("  processing…")
    time.sleep(5)
    video = client.files.get(name=video.name)
if video.state.name == "FAILED":
    raise RuntimeError("Gemini could not process this video. Try a shorter clip or mp4 format.")
print("✅ Video ready")

## Step 4 · Analyze the workout
Edit the prompt and re-run just this cell to ask something different — no need to re-upload.

In [ ]:
from IPython.display import Markdown, display

prompt = """You are an expert fitness coach. Analyze this workout video:
1. List each exercise performed with start/end timestamps (MM:SS).
2. Count the reps in each set.
3. Point out form issues (with timestamps) and give specific coaching tips.
4. Estimate total active time vs rest time.
5. Give an overall score out of 10 and the top 3 things to improve.
Start with a short summary, then give a table of exercises/sets/reps."""

response = client.models.generate_content(model=MODEL, contents=[video, prompt])
display(Markdown(response.text))

## Step 5 · (Optional) Delete the video from Gemini
Uploaded files auto-delete after 48 hours anyway.

In [ ]:
client.files.delete(name=video.name)
print("🗑️ Deleted")